# LangChain

In [2]:
import os

from dotenv import load_dotenv
from groq import Groq
from langchain.chat_models import init_chat_model
from langchain_ollama import ChatOllama

load_dotenv()

True

In [3]:
# List available models
sorted(m.id for m in Groq().models.list().data)

['allam-2-7b',
 'canopylabs/orpheus-arabic-saudi',
 'canopylabs/orpheus-v1-english',
 'meta-llama/llama-prompt-guard-2-22m',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-120b',
 'openai/gpt-oss-20b',
 'openai/gpt-oss-safeguard-20b',
 'qwen/qwen3.8-27b',
 'whisper-large-v3',
 'whisper-large-v3-turbo']

In [13]:
# Simple request
model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0,
    max_tokens=1000,
    reasoning_effort="high",
)

response = model.invoke("What is a sensor dropout? Answer in one sentence.")

if response.response_metadata.get("finish_reason") == "length":
    raise RuntimeError(
        f"Response truncated by max_tokens. Usage: {response.usage_metadata}"
    )

print(type(response).__name__)
print("content: ", response.content)
print("usage: ", response.usage_metadata)
print("metadata: ", response.response_metadata)
print("finish_reason:", response.response_metadata.get("finish_reason"))
print("extras: ", response.additional_kwargs)

AIMessage
content:  A sensor dropout is a brief interruption or loss of data from a sensor, typically caused by signal interference, power issues, or hardware faults.
usage:  {'input_tokens': 82, 'output_tokens': 136, 'total_tokens': 218, 'output_token_details': {'reasoning': 99}}
metadata:  {'token_usage': {'completion_tokens': 136, 'prompt_tokens': 82, 'total_tokens': 218, 'completion_time': 0.144994791, 'completion_tokens_details': {'reasoning_tokens': 99}, 'prompt_time': 0.004072425, 'prompt_tokens_details': None, 'queue_time': 0.046767575, 'total_time': 0.149067216}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_feb9b278f1', 'service_tier': 'on_demand', 'reasoning_effort': 'high', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}
finish_reason: stop
extras:  {'reasoning_content': 'The user asks: "What is a sensor dropout? Answer in one sentence." So we need to give a concise definition. A sensor dropout is a temporary loss of signal or data from 

**[Notes]**
- `Temperature = 0`: always the most likely (stable responses, good for agents).
- `Temperature > 0`: Higher values = greater variation (good for creative text).

In [8]:
model = init_chat_model(
    os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    model_provider="groq",
    temperature=0,
    max_tokens=1000,
    reasoning_effort="low",
)

resposta = model.invoke([
    {
        "role": "system",
        "content": "Você é engenheiro de confiabilidade industrial. Responda em 2 frases."
    },
    {
        "role": "user",
        "content": "O que é um dropout de sensor?"
    },
])

print(type(resposta).__name__)
print("content: ", resposta.content)
print("usage:", resposta.usage_metadata)
print("finish_reason:", resposta.response_metadata.get("finish_reason"))
print("reasoning:", resposta.additional_kwargs)

AIMessage
content:  Um dropout de sensor ocorre quando o dispositivo deixa de enviar ou registrar dados por um curto intervalo, criando lacunas no histórico de medições. Isso pode ser causado por falhas de comunicação, alimentação instável ou problemas internos do próprio sensor.
usage: {'input_tokens': 99, 'output_tokens': 67, 'total_tokens': 166, 'output_token_details': {'reasoning': 10}}
finish_reason: stop
reasoning: {'reasoning_content': 'Need answer in Portuguese, 2 sentences.'}


finish_reason: length
usage: {'input_tokens': 82, 'output_tokens': 30, 'total_tokens': 112, 'output_token_details': {'reasoning': 28}}
reasoning: {'reasoning_content': 'The user asks: "What is a sensor dropout? Answer in one sentence." So we need to give a concise definition. A sensor'}


## Ollama

Running it locally

In [34]:
llm = ChatOllama(
    model="llama3.2",
    base_url=f"http://{os.getenv('OLLAMA_HOST')}:{os.getenv('OLLAMA_PORT')}",
    temperature=50
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    (
        "human", "I love programming."
    ),
]

ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='J\'adore le programme informatique.\n\n(Note: "J\'adore" means "I love" and "programme informatique" means "information computer", or more commonly, "code", or simply "programe" when used in this context.)\n\nIf you want the translation in an idiomatic expression or common way that people in French usually use to say "I love coding" or "I love programming" it would be :\n\n Je aime la programmation.\n\nLet me know if you want the translation for "the code" as the whole "I love coding", because "J\'adore le programme informatique" will make people believe that you just talk about a physical program or the software.', additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-10-27T19:26:14.051639591Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11380274977, 'load_duration': 119382065, 'prompt_eval_count': 45, 'prompt_eval_duration': 76760423, 'eval_count': 145, 'eval_duration': 11139599574, 'logprobs': None, 'model_name': 'llama3.2',

In [33]:
print(type(ai_msg).__name__)
print("content: ", ai_msg.content)
print("usage: ", ai_msg.usage_metadata)
print("metadata: ", ai_msg.response_metadata)
print("extras: ", ai_msg.additional_kwargs)

AIMessage
content:  J'aime la programmation.

(Note: I've used the formal form "je" instead of the informal form "je l'aime" because you didn't indicate your familiarity or friendship with the recipient of the sentence.)
usage:  {'input_tokens': 45, 'output_tokens': 46, 'total_tokens': 91}
metadata:  {'model': 'llama3.2', 'created_at': '2026-10-27T19:25:43.906473149Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3722305594, 'load_duration': 121738330, 'prompt_eval_count': 45, 'prompt_eval_duration': 77374735, 'eval_count': 46, 'eval_duration': 3507793859, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}
extras:  {}


___